# Detecção de Fraudes em Transações

Notebook didático que reutiliza os módulos de `src`, evitando duplicação. Execute a partir da raiz do projeto. Métricas só aparecem após a execução com o dataset real.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
from src.config import DATA_PATH, FIGURES_DIR, METRICS_DIR, ensure_directories
from src.data_processing import load_data, validate_data, create_eda_figures, split_data
from src.feature_engineering import create_features
from src.train import build_candidates, train_and_compare, choose_best
from src.evaluate import threshold_analysis, calculate_metrics
ensure_directories()

## Carregamento e análise exploratória
O carregador valida esquema, alvo, tipos, ausentes e duplicados. Os gráficos são salvos em `reports/figures`.

In [ ]:
df = load_data(DATA_PATH)
validate_data(df)

In [ ]:
create_eda_figures(df)
create_features(df.drop(columns='Class')).head()

## Divisão, modelos e balanceamento
A divisão 70/15/15 ocorre antes de scaler ou reamostragem. SMOTE permanece dentro do pipeline e nunca vê validação/teste.

In [ ]:
x_train, x_val, x_test, y_train, y_val, y_test = split_data(df)
fitted, comparison = train_and_compare(build_candidates(y_train), x_train, y_train, x_val, y_val)
comparison

## Threshold e avaliação
O threshold é escolhido exclusivamente na validação. Consulte o teste somente depois da decisão final.

In [ ]:
best_name = choose_best(comparison)
best = fitted[best_name]
val_prob = best.predict_proba(x_val)[:, 1]
threshold_analysis(y_val, val_prob)

## SHAP e conclusão
Execute `python main.py --explain` para gerar explicações globais e locais. SHAP explica contribuição, não causalidade. A conclusão deve usar os artefatos reais e discutir precisão, recall, PR-AUC, custo, tempo, interpretabilidade e overfitting.